# Tutorial：用 Whisper-small 做粤语 ASR Full SFT

**Audience**：第一次参与语音识别竞赛、已经了解基本深度学习训练循环的队员。

**Prerequisites**：服务器环境已按项目 README 部署；官方数据、manifest 和 Whisper-small 已保存在服务器本地。

**Learning goals**：理解音频→Log-Mel→Whisper→粤语 token 的数据流；检查官方数据；运行短 smoke test；阅读训练报告。正式长训练请使用 `train.py` 与 Slurm，不要在 Notebook 内运行。

## Outline

1. 加载固定路径与 manifest
2. 抽查音频、粤语标签和场景
3. 查看 Whisper 输入特征与官方指标
4. 理解 Full SFT 配置和 smoke test
5. 阅读固定脚本生成的实验图表

In [ ]:
from __future__ import annotations

import json
import random
import sys
from pathlib import Path

SEED = 42
random.seed(SEED)
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'train.py').is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
MANIFEST_DIR = PROJECT_ROOT / 'artifacts/manifests'
MODEL_DIR = PROJECT_ROOT / 'artifacts/models/whisper-small'
PROJECT_ROOT

## 1. 加载经过质检的 manifest

`train.jsonl` 用于更新权重；`train_probe.jsonl` 是训练集中的固定 256 条诊断样本；`validation.jsonl` 才用于 checkpoint 和模型选择。公开 `template_pre` 不参与这一步。

In [ ]:
from cantonese_asr.io import read_jsonl

train_rows = read_jsonl(MANIFEST_DIR / 'train.jsonl')
validation_rows = read_jsonl(MANIFEST_DIR / 'validation.jsonl')
probe_rows = read_jsonl(MANIFEST_DIR / 'train_probe.jsonl')
data_report = json.loads((MANIFEST_DIR / 'data_report.json').read_text(encoding='utf-8'))
{
    'train': len(train_rows),
    'validation': len(validation_rows),
    'train_probe': len(probe_rows),
    'text_overlap': data_report['leakage_checks']['normalized_text_overlap'],
    'hours': data_report['duration_s']['total_hours'],
}

## 2. 随机播放并核对一条训练样本

听到的内容必须与 `text` 的粤语原文一致。普通话翻译和粤拼只帮助人工检查，不作为本轮 SFT 标签。

In [ ]:
from IPython.display import Audio, display

sample = random.choice(train_rows)
audio_path = Path(sample['audio_path'])
if not audio_path.is_absolute():
    audio_path = PROJECT_ROOT / audio_path
print({key: sample.get(key) for key in ('id', 'text', 'mandarin', 'jyutping', 'scene', 'duration_s')})
display(Audio(filename=str(audio_path)))

## 3. 查看 16 kHz 音频与 Log-Mel 输入

Whisper 的 feature extractor 会把波形转换成固定形式的 Log-Mel 特征。Full SFT 不会修改这一架构；训练改变的是模型的可学习参数。

In [ ]:
import librosa
import matplotlib.pyplot as plt
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained(
    MODEL_DIR, language='zh', task='transcribe', local_files_only=True
)
waveform, _ = librosa.load(audio_path, sr=16_000, mono=True)
features = processor.feature_extractor(waveform, sampling_rate=16_000).input_features[0]
fig, ax = plt.subplots(figsize=(12, 4))
image = ax.imshow(features, aspect='auto', origin='lower')
ax.set(title='Whisper log-Mel input', xlabel='Frame', ylabel='Mel bin')
fig.colorbar(image, ax=ax)
plt.show()

## 4. 验证官方比赛指标

正式主指标不是普通的逐字 accuracy。预测先繁转简、去标点和空白，再计算字符编辑距离；距离不超过 2 的整句计为正确。CER 是保护指标。

In [ ]:
from cantonese_asr.metrics import compute_official_metrics

references = ['我今日想去饮茶。', '天气很好！']
predictions = ['我今日去飲茶', '天气糟糕啊']
compute_official_metrics(references, predictions)

## 5. Full SFT 与 smoke test

正式训练使用 AdamW、warmup 和 linear/cosine scheduler。先用 16–32 条样本尝试过拟合，是为了证明音频、标签、反向传播、保存恢复和推理全部连接正确，不代表最终超参数。

In [ ]:
smoke_command = '''sbatch slurm/smoke.slurm'''
grid_command = '''BATCH_SIZE=4 GRAD_ACCUM=4 bash scripts/submit_grid.sh'''
print('Smoke test:', smoke_command)
print('Four independent 4090 trials:', grid_command)

## 6. 读取固定绘图脚本生成的报告

每轮结束后运行 `scripts/plot_experiments.py`。它读取标准 `metrics.jsonl`，不需要重新写绘图代码；最佳模型始终只看 validation。

In [ ]:
from IPython.display import Image

report_dir = PROJECT_ROOT / 'artifacts/reports/experiments/all'
scoreboard_path = report_dir / 'scoreboard.json'
if scoreboard_path.is_file():
    display(json.loads(scoreboard_path.read_text(encoding='utf-8')))
    display(Image(filename=str(report_dir / 'training_curves.png')))
else:
    print('尚无实验报告；训练完成后由固定脚本生成。')

## Exercise

1. 找出 validation 中时长最长的 5 条样本。
2. 比较 train-probe 与 validation 的整句准确率差距；差距持续扩大说明什么？
3. 常见错误：把公开 `template_pre` 当调参集。修复方式：只用固定 internal validation 选模。

In [ ]:
# Exercise answer scaffold
longest = sorted(validation_rows, key=lambda row: row['duration_s'], reverse=True)[:5]
[(row['id'], row['duration_s'], row['text']) for row in longest]

## Pitfalls and extensions

- OOM：降低 micro-batch，同时提高 gradient accumulation，保持 effective batch。
- Loss 下降但 validation 不升：检查过拟合差距、数据连接和文本规范化。
- 可选扩展：完成官方 Full SFT baseline 后，才单独评估合法外部数据或 SenseVoice 教师蒸馏。